# Experimentación: CNN sobre CIFAR-10
**Evaluación Segunda Unidad - Inteligencia Artificial (UAC)** · Semilla individual: **75**

Este notebook reúne las pruebas de la Pregunta 1 (carga, EDA, visualización, transformaciones, partición) y la validación de shapes de la Pregunta 3.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import torch
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

from dataset import *   # SEED, CLASSES, MEAN, STD, set_seed, get_transforms, get_datasets, show_samples
set_seed(SEED)
print("Semilla individual:", SEED)
print("torch:", torch.__version__)

## 1. Carga del dataset y EDA

In [ ]:
raw_train = datasets.CIFAR10(DATA_DIR, train=True, download=True)
raw_test = datasets.CIFAR10(DATA_DIR, train=False, download=True)
print(f"Entrenamiento: {len(raw_train)} imágenes | Prueba: {len(raw_test)} imágenes")
print(f"Tamaño de cada imagen (PIL): {raw_train[0][0].size} | modo: {raw_train[0][0].mode}")
print(f"Clases: {CLASSES}")

In [ ]:
# Distribución de clases (el dataset está balanceado)
counts = np.bincount(raw_train.targets)
for c, n in zip(CLASSES, counts):
    print(f"{c:<10} {n}")

plt.figure(figsize=(8, 3.5))
plt.bar(CLASSES, counts, color="steelblue")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Imágenes")
plt.title("Distribución de clases - CIFAR-10 (entrenamiento)")
plt.tight_layout()
plt.show()

In [ ]:
# Estadísticas por canal RGB (justifican la normalización)
data = np.asarray(raw_train.data, dtype=np.float32) / 255.0   # (50000, 32, 32, 3)
print("Shape del arreglo:", data.shape)
print("Media por canal RGB:", data.mean(axis=(0, 1, 2)).round(4), "-> MEAN usado:", MEAN)
print("Desv. estándar por canal:", data.std(axis=(0, 1, 2)).round(4), "-> STD usado:", STD)

## 2. Visualización de muestras (16 imágenes con su etiqueta real)

In [ ]:
show_samples(show=True)

## 3. Pipeline de transformaciones y Data Augmentation

In [ ]:
train_tf, eval_tf = get_transforms()
print("Transformaciones de entrenamiento:\n", train_tf)
print("\nTransformaciones de validación/prueba:\n", eval_tf)

In [ ]:
# Efecto del aumento de datos sobre una misma imagen
img, label = raw_train[0]
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
aug = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(p=0.5)])
for ax in axes[1:]:
    ax.imshow(aug(img)); ax.set_title("Aumentada"); ax.axis("off")
fig.suptitle(f"Data Augmentation sobre una imagen de clase '{CLASSES[label]}'")
plt.tight_layout(); plt.show()

## 4. Partición estratificada train / validación

In [ ]:
train_set, val_set, test_set = get_datasets()

targets = np.array(raw_train.targets)
tr_counts = np.bincount(targets[train_set.indices], minlength=10)
va_counts = np.bincount(targets[val_set.indices], minlength=10)
print(f"{'Clase':<10}{'Train':>8}{'Val':>8}{'% Val':>9}")
for c, a, b in zip(CLASSES, tr_counts, va_counts):
    print(f"{c:<10}{a:>8}{b:>8}{100 * b / (a + b):>8.1f}%")
assert set(train_set.indices).isdisjoint(val_set.indices), "Fuga de datos entre train y val"
print("\nOK: partición estratificada (10 % por clase) y sin solapamiento.")

In [ ]:
x, y = next(iter(torch.utils.data.DataLoader(train_set, batch_size=64)))
print("Batch de entrenamiento ->", tuple(x.shape), "(B, C, H, W) | etiquetas:", tuple(y.shape))
print("Rango tras normalizar: min {:.2f}, max {:.2f}".format(x.min().item(), x.max().item()))

## 5. Validación de la arquitectura y de los tensores 4D (Pregunta 3)

In [ ]:
from model import CNN, validate_shapes, count_parameters
model = validate_shapes(batch_size=8)
print(model)

In [ ]:
# Prueba de cordura: la red debe poder memorizar un único lote (la loss baja claramente)
import torch.nn as nn, torch.optim as optim
set_seed(SEED)
net = CNN()
crit, opt = nn.CrossEntropyLoss(), optim.Adam(net.parameters(), lr=1e-3)
xb, yb = x[:32], y[:32]
net.train()
for step in range(30):
    opt.zero_grad(); loss = crit(net(xb), yb); loss.backward(); opt.step()
    if step % 10 == 0 or step == 29:
        print(f"paso {step:02d} | loss = {loss.item():.4f}")

## 6. Resultados del entrenamiento completo
Ejecuta `python src/train.py` desde la raíz del repositorio para generar las gráficas y métricas; luego vuelve a ejecutar esta celda.

In [ ]:
from IPython.display import Image, display
for name in ("loss_accuracy.png", "confusion_matrix.png"):
    path = OUT_DIR / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"Aún no existe {name}: ejecuta python src/train.py")